In [ ]:
import os
import json
import glob
import math
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional stats (script will still run if scipy not installed)
try:
    from scipy.stats import spearmanr, pearsonr, ttest_ind
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False


# ----------------------------
# Config
# ----------------------------
RESULT_DIR = "./result"
OUT_DIR = "./analysis_out"

FIG_DIR = os.path.join(OUT_DIR, "figures")
HEAT_DIR = os.path.join(FIG_DIR, "heatmaps")
TRIAL_HEAT_DIR = os.path.join(HEAT_DIR, "per_trial")

EVENTS = ["Stable", "SizeChange", "Merge", "Split"]
W_VALUES = [1, 2, 5, 10, 20, 50]

# nodelink viewport size (your SVG container)
VIEW_W = 700
VIEW_H = 700

# heatmap bins (coarser bins are more robust)
HEAT_BINS = 70  # 70x70 => 10px per bin (since 700/70=10)

# node-attention parameters
NEAR_RADIUS_PX = 35  # how close a mouse point counts as "near a node"
HIGH_ENTROPY_Q = 0.75  # top 25% nodes in a trial are "high entropy"

# H_alias bins for grouping
H_ALIAS_BINS = [
    ("low", 0.0, 0.3),
    ("mid", 0.3, 0.7),
    ("high", 0.7, 10.0),
]


def ensure_dirs():
    for p in [OUT_DIR, FIG_DIR, HEAT_DIR, TRIAL_HEAT_DIR]:
        os.makedirs(p, exist_ok=True)


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def load_all_results(result_dir: str) -> Tuple[pd.DataFrame, List[dict]]:
    """
    Loads all JSON result files. Each file is expected to be a list[trial_record].
    Returns:
      - flat dataframe (one row per trial)
      - raw trials (list of dicts, flattened across files)
    """
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    if not paths:
        raise FileNotFoundError(f"No result JSON files found in {result_dir}")

    all_trials = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            try:
                trials = json.load(f)
            except Exception as e:
                raise RuntimeError(f"Failed to parse JSON: {p} ({e})")
        if not isinstance(trials, list):
            raise ValueError(f"Result file should be a list of trials: {p}")
        for t in trials:
            t["_source_file"] = os.path.basename(p)
            all_trials.append(t)

    # Flatten into dataframe with robust field access
    rows = []
    for t in all_trials:
        rows.append({
            "source_file": t.get("_source_file"),
            "subject_id": t.get("subject_id"),
            "trial_index": t.get("trial_index"),
            "W": t.get("W"),
            "dataset_tag": t.get("dataset_tag"),
            "target_community": t.get("target_community"),
            "target_name": t.get("target_name"),
            "ground_truth": t.get("ground_truth"),
            "answer": t.get("answer"),
            "correct": t.get("correct"),
            "confidence": t.get("confidence"),
            "duration_ms": t.get("duration_ms"),
            "space_toggle_count": t.get("space_toggle_count"),
            "mouse_entropy": t.get("mouse_entropy"),
            "h_alias": t.get("h_alias"),
            "delta_sigma": t.get("delta_sigma"),
            "curr_snapshot_id": t.get("curr_snapshot_id"),
            "prev_snapshot_id": t.get("prev_snapshot_id"),
        })

    df = pd.DataFrame(rows)

    # type fixes
    for c in ["trial_index", "W", "target_community", "correct", "confidence",
              "duration_ms", "space_toggle_count", "mouse_entropy", "h_alias", "delta_sigma",
              "curr_snapshot_id", "prev_snapshot_id"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # normalize event names (guard older versions)
    df["ground_truth"] = df["ground_truth"].astype(str)
    df["answer"] = df["answer"].astype(str)

    return df, all_trials


def add_halias_bin(df: pd.DataFrame) -> pd.DataFrame:
    def bin_one(h):
        if pd.isna(h):
            return "unknown"
        for name, lo, hi in H_ALIAS_BINS:
            if (h >= lo) and (h < hi):
                return name
        return "unknown"

    df = df.copy()
    df["h_alias_bin"] = df["h_alias"].apply(bin_one)
    return df


def summarize_subjects(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("subject_id", dropna=False)
    out = g.agg(
        n_trials=("correct", "count"),
        acc=("correct", "mean"),
        rt_mean=("duration_ms", "mean"),
        rt_median=("duration_ms", "median"),
        conf_mean=("confidence", "mean"),
        toggles_mean=("space_toggle_count", "mean"),
        mouse_entropy_mean=("mouse_entropy", "mean"),
    ).reset_index()
    return out.sort_values("acc")


def confusion_matrix(df: pd.DataFrame, normalize: bool = True) -> pd.DataFrame:
    # ensure ordering
    gt = df["ground_truth"].astype(str)
    ans = df["answer"].astype(str)
    cm = pd.crosstab(gt, ans, dropna=False)

    # enforce consistent columns/rows
    for e in EVENTS:
        if e not in cm.index:
            cm.loc[e] = 0
        if e not in cm.columns:
            cm[e] = 0
    cm = cm.loc[EVENTS, EVENTS]

    if normalize:
        cm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)
    return cm


def top_misclassifications(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each ground_truth, list top wrong answers.
    """
    wrong = df[df["correct"] == 0].copy()
    if wrong.empty:
        return pd.DataFrame(columns=["ground_truth", "most_confused_with", "count", "ratio_within_gt_wrong"])

    rows = []
    for gt, sub in wrong.groupby("ground_truth"):
        counts = sub["answer"].value_counts()
        total = counts.sum()
        for ans, cnt in counts.head(3).items():
            rows.append({
                "ground_truth": gt,
                "most_confused_with": ans,
                "count": int(cnt),
                "ratio_within_gt_wrong": float(cnt / total) if total else np.nan
            })
    return pd.DataFrame(rows)


def save_table(df: pd.DataFrame, name: str):
    path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"[saved] {path}")


def plot_bar(df: pd.DataFrame, x: str, y: str, title: str, outname: str):
    plt.figure()
    tmp = df[[x, y]].dropna()
    plt.bar(tmp[x].astype(str), tmp[y].astype(float))
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=30)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, outname)
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"[saved] {path}")


def plot_line(df: pd.DataFrame, x: str, y: str, group: Optional[str], title: str, outname: str):
    plt.figure()
    tmp = df[[x, y] + ([group] if group else [])].dropna()
    if group:
        for g, sub in tmp.groupby(group):
            xs = sub[x].astype(float).values
            ys = sub[y].astype(float).values
            order = np.argsort(xs)
            plt.plot(xs[order], ys[order], marker="o", label=str(g))
        plt.legend()
    else:
        xs = tmp[x].astype(float).values
        ys = tmp[y].astype(float).values
        order = np.argsort(xs)
        plt.plot(xs[order], ys[order], marker="o")
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, outname)
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"[saved] {path}")


def plot_heatmap(mat: np.ndarray, title: str, outpath: str):
    plt.figure()
    plt.imshow(mat.T, origin="lower", aspect="auto")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


def extract_mouse_points(trial: dict) -> np.ndarray:
    """
    Returns Nx2 array of (x,y) in screen coords for trajectory_sample.
    Filters null points; clamps to viewport.
    """
    traj = trial.get("trajectory_sample") or []
    pts = []
    for p in traj:
        x = p.get("x", None)
        y = p.get("y", None)
        if x is None or y is None:
            continue
        try:
            xf = float(x)
            yf = float(y)
        except Exception:
            continue
        # keep only points inside viewport
        if 0 <= xf <= VIEW_W and 0 <= yf <= VIEW_H:
            pts.append((xf, yf))
    if not pts:
        return np.zeros((0, 2), dtype=float)
    return np.array(pts, dtype=float)


def heatmap_from_trials(trials: List[dict], mask_fn=None) -> np.ndarray:
    """
    Aggregate 2D histogram heatmap (screen space).
    """
    all_pts = []
    for t in trials:
        if mask_fn and not mask_fn(t):
            continue
        pts = extract_mouse_points(t)
        if len(pts) > 0:
            all_pts.append(pts)
    if not all_pts:
        return np.zeros((HEAT_BINS, HEAT_BINS), dtype=float)

    P = np.vstack(all_pts)
    # histogram2d: x then y
    H, _, _ = np.histogram2d(P[:, 0], P[:, 1], bins=HEAT_BINS, range=[[0, VIEW_W], [0, VIEW_H]])
    # log scale for visibility
    return np.log1p(H)


def trial_node_attention_stats(trial: dict) -> Optional[dict]:
    """
    Uses layout_screen (node positions) + curr_nodes entropy + mouse points
    to quantify whether mouse points concentrate near high-entropy nodes.
    """
    layout = trial.get("layout_screen")
    nodes = trial.get("curr_nodes")

    if not isinstance(layout, dict) or not isinstance(nodes, list):
        return None

    pts = extract_mouse_points(trial)
    if len(pts) == 0:
        return None

    # node positions
    node_ids = []
    pos = []
    ent = []
    for nd in nodes:
        nid = nd.get("id")
        if nid is None:
            continue
        key = str(nid)
        if key not in layout:
            continue
        node_ids.append(int(nid))
        pos.append([layout[key]["x"], layout[key]["y"]])
        ent.append(safe_float(nd.get("entropy")))
    if len(pos) == 0:
        return None

    pos = np.array(pos, dtype=float)  # Mx2
    ent = np.array(ent, dtype=float)  # M

    # define high entropy nodes by quantile
    q = np.nanquantile(ent, HIGH_ENTROPY_Q) if np.isfinite(ent).any() else np.nan
    high_mask = ent >= q if np.isfinite(q) else np.zeros_like(ent, dtype=bool)

    # For each mouse point, find nearest node distance
    # Compute distance matrix (Npts x Mnodes)
    d2 = ((pts[:, None, :] - pos[None, :, :]) ** 2).sum(axis=2)
    nearest_idx = np.argmin(d2, axis=1)
    nearest_dist = np.sqrt(d2[np.arange(len(pts)), nearest_idx])

    near = nearest_dist <= NEAR_RADIUS_PX
    if near.sum() == 0:
        # no near-node interactions
        near_high_ratio = 0.0
        corr = np.nan
    else:
        near_nodes = nearest_idx[near]
        near_high = high_mask[near_nodes].sum()
        near_high_ratio = float(near_high / len(near_nodes))

        # attention counts per node (near only)
        att = np.zeros(len(pos), dtype=float)
        for idx in near_nodes:
            att[idx] += 1.0

        # correlation between node entropy and attention
        if SCIPY_OK and np.isfinite(ent).all() and att.sum() > 0:
            corr = spearmanr(ent, att).correlation
        else:
            # fallback: Pearson on finite values
            finite = np.isfinite(ent)
            if finite.sum() >= 3 and att.sum() > 0:
                corr = np.corrcoef(ent[finite], att[finite])[0, 1]
            else:
                corr = np.nan

    return {
        "near_points": int(near.sum()),
        "total_points": int(len(pts)),
        "near_high_entropy_ratio": near_high_ratio,
        "entropy_attention_corr": corr,
        "high_entropy_threshold_q": float(q) if np.isfinite(q) else np.nan,
    }


def make_trial_heatmap_png(trial: dict, outpath: str):
    pts = extract_mouse_points(trial)
    if len(pts) == 0:
        return
    H, _, _ = np.histogram2d(pts[:, 0], pts[:, 1], bins=HEAT_BINS, range=[[0, VIEW_W], [0, VIEW_H]])
    H = np.log1p(H)
    title = f"Trial heatmap | GT={trial.get('ground_truth')} | W={trial.get('W')} | h_alias={trial.get('h_alias'):.3f}"
    plot_heatmap(H, title, outpath)


def confidence_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns per-confidence accuracy and counts, for calibration-like analysis.
    """
    tmp = df.dropna(subset=["confidence", "correct"]).copy()
    tmp["confidence"] = tmp["confidence"].astype(int)
    g = tmp.groupby("confidence")
    out = g.agg(
        n=("correct", "count"),
        acc=("correct", "mean"),
        rt=("duration_ms", "mean"),
    ).reset_index().sort_values("confidence")
    return out


def main():
    ensure_dirs()

    df, raw_trials = load_all_results(RESULT_DIR)
    df = add_halias_bin(df)

    # ----------------------------
    # Save raw merged table
    # ----------------------------
    save_table(df, "trials_merged")

    # ----------------------------
    # Subject-level summary
    # ----------------------------
    subj = summarize_subjects(df)
    save_table(subj, "subjects_summary")

    # ----------------------------
    # Core descriptive stats
    # ----------------------------
    overall = pd.DataFrame([{
        "n_subjects": df["subject_id"].nunique(),
        "n_trials": len(df),
        "overall_acc": df["correct"].mean(),
        "rt_mean_ms": df["duration_ms"].mean(),
        "conf_mean": df["confidence"].mean(),
        "space_toggle_mean": df["space_toggle_count"].mean(),
        "mouse_entropy_mean": df["mouse_entropy"].mean(),
        "h_alias_mean": df["h_alias"].mean(),
        "delta_sigma_mean": df["delta_sigma"].mean(),
    }])
    save_table(overall, "overall_summary")

    # ----------------------------
    # Accuracy / RT by W, event, h_alias_bin
    # ----------------------------
    byW = df.groupby("W").agg(
        n=("correct", "count"),
        acc=("correct", "mean"),
        rt=("duration_ms", "mean"),
        conf=("confidence", "mean"),
        toggles=("space_toggle_count", "mean"),
        mouse_entropy=("mouse_entropy", "mean"),
        h_alias=("h_alias", "mean"),
    ).reset_index().sort_values("W")
    save_table(byW, "by_W")

    byEvent = df.groupby("ground_truth").agg(
        n=("correct", "count"),
        acc=("correct", "mean"),
        rt=("duration_ms", "mean"),
        conf=("confidence", "mean"),
        toggles=("space_toggle_count", "mean"),
        mouse_entropy=("mouse_entropy", "mean"),
        h_alias=("h_alias", "mean"),
    ).reset_index()
    save_table(byEvent, "by_event")

    byBin = df.groupby("h_alias_bin").agg(
        n=("correct", "count"),
        acc=("correct", "mean"),
        rt=("duration_ms", "mean"),
        conf=("confidence", "mean"),
        toggles=("space_toggle_count", "mean"),
        mouse_entropy=("mouse_entropy", "mean"),
    ).reset_index()
    save_table(byBin, "by_halias_bin")

    # Plots
    plot_line(byW, "W", "acc", None, "Accuracy vs W", "acc_vs_W.png")
    plot_line(byW, "W", "rt", None, "Reaction time (ms) vs W", "rt_vs_W.png")
    plot_bar(byEvent.sort_values("acc"), "ground_truth", "acc", "Accuracy by event type", "acc_by_event.png")
    plot_bar(byBin, "h_alias_bin", "acc", "Accuracy by H_alias bin", "acc_by_halias_bin.png")

    # ----------------------------
    # Confusion matrix + top confusions
    # ----------------------------
    cm = confusion_matrix(df, normalize=True)
    cm_path = os.path.join(OUT_DIR, "confusion_matrix_norm.csv")
    cm.to_csv(cm_path)
    print(f"[saved] {cm_path}")

    top_conf = top_misclassifications(df)
    save_table(top_conf, "top_misclassifications")

    # Make a confusion heatmap figure
    plt.figure()
    plt.imshow(cm.values, origin="upper", aspect="auto")
    plt.xticks(range(len(EVENTS)), EVENTS, rotation=30)
    plt.yticks(range(len(EVENTS)), EVENTS)
    plt.title("Confusion matrix (row-normalized)")
    plt.tight_layout()
    p = os.path.join(FIG_DIR, "confusion_matrix_norm.png")
    plt.savefig(p, dpi=220)
    plt.close()
    print(f"[saved] {p}")

    # ----------------------------
    # Confidence usefulness
    # ----------------------------
    conf_tbl = confidence_analysis(df)
    save_table(conf_tbl, "confidence_accuracy_table")
    plot_line(conf_tbl, "confidence", "acc", None, "Accuracy vs confidence", "acc_vs_confidence.png")
    plot_line(conf_tbl, "confidence", "rt", None, "RT vs confidence", "rt_vs_confidence.png")

    # Correct vs incorrect confidence
    ci = df.dropna(subset=["confidence", "correct"]).copy()
    corr_conf = ci[ci["correct"] == 1]["confidence"].mean()
    wrong_conf = ci[ci["correct"] == 0]["confidence"].mean()
    with open(os.path.join(OUT_DIR, "confidence_correct_vs_wrong.txt"), "w", encoding="utf-8") as f:
        f.write(f"mean_confidence_correct = {corr_conf}\n")
        f.write(f"mean_confidence_wrong   = {wrong_conf}\n")
    print("[saved] confidence_correct_vs_wrong.txt")

    # ----------------------------
    # Heatmaps (screen space)
    # ----------------------------
    # Overall heatmap
    H_all = heatmap_from_trials(raw_trials)
    plot_heatmap(H_all, "Mouse heatmap (all trials, log1p counts)", os.path.join(HEAT_DIR, "heat_all.png"))

    # Heatmap by event type
    for ev in EVENTS:
        H = heatmap_from_trials(raw_trials, mask_fn=lambda t, ev=ev: str(t.get("ground_truth")) == ev)
        plot_heatmap(H, f"Mouse heatmap | ground_truth={ev}", os.path.join(HEAT_DIR, f"heat_event_{ev}.png"))

    # Heatmap by H_alias bin
    def halias_bin_of_trial(t):
        h = t.get("h_alias", None)
        try:
            h = float(h)
        except Exception:
            return "unknown"
        for name, lo, hi in H_ALIAS_BINS:
            if lo <= h < hi:
                return name
        return "unknown"

    for name, _, _ in H_ALIAS_BINS:
        H = heatmap_from_trials(raw_trials, mask_fn=lambda t, name=name: halias_bin_of_trial(t) == name)
        plot_heatmap(H, f"Mouse heatmap | H_alias bin={name}", os.path.join(HEAT_DIR, f"heat_halias_{name}.png"))

    # ----------------------------
    # Node-attention vs entropy (Does mouse focus on high-entropy nodes?)
    # ----------------------------
    att_rows = []
    for t in raw_trials:
        st = trial_node_attention_stats(t)
        if st is None:
            continue
        att_rows.append({
            "source_file": t.get("_source_file"),
            "subject_id": t.get("subject_id"),
            "trial_index": t.get("trial_index"),
            "W": t.get("W"),
            "ground_truth": t.get("ground_truth"),
            "h_alias": t.get("h_alias"),
            "delta_sigma": t.get("delta_sigma"),
            **st
        })
    att_df = pd.DataFrame(att_rows)
    if len(att_df) > 0:
        save_table(att_df, "node_attention_stats")

        # Plot: near_high_entropy_ratio vs h_alias (binned)
        att_df2 = att_df.copy()
        att_df2["h_alias_bin"] = pd.cut(
            att_df2["h_alias"].astype(float),
            bins=[0, 0.3, 0.7, 10],
            labels=["low", "mid", "high"],
            include_lowest=True
        )
        att_bin = att_df2.groupby("h_alias_bin").agg(
            n=("near_high_entropy_ratio", "count"),
            near_high_ratio=("near_high_entropy_ratio", "mean"),
            corr=("entropy_attention_corr", "mean"),
        ).reset_index()
        save_table(att_bin, "node_attention_by_halias_bin")

        plot_bar(att_bin, "h_alias_bin", "near_high_ratio",
                 "P(mouse near high-entropy nodes) by H_alias bin", "near_high_ratio_by_halias_bin.png")

        # Optional stats
        if SCIPY_OK:
            low = att_df2[att_df2["h_alias_bin"] == "low"]["near_high_entropy_ratio"].dropna().values
            high = att_df2[att_df2["h_alias_bin"] == "high"]["near_high_entropy_ratio"].dropna().values
            if len(low) >= 10 and len(high) >= 10:
                tstat = ttest_ind(high, low, equal_var=False)
                with open(os.path.join(OUT_DIR, "node_attention_stats_test.txt"), "w", encoding="utf-8") as f:
                    f.write("Welch t-test: near_high_entropy_ratio (high vs low H_alias)\n")
                    f.write(str(tstat) + "\n")
                print("[saved] node_attention_stats_test.txt")

    # ----------------------------
    # Per-trial heatmaps for the "most informative" trials
    #   - Top-K hardest (lowest accuracy)
    # ----------------------------
    # We define a "trial identity" for grouping across subjects.
    # If your system fixed the same underlying trials for everyone, then (W, curr_snapshot_id, ground_truth, target_community)
    # is a good key.
    df_key = df.copy()
    df_key["trial_key"] = (
        df_key["W"].astype("Int64").astype(str) + "|" +
        df_key["curr_snapshot_id"].astype("Int64").astype(str) + "|" +
        df_key["ground_truth"].astype(str) + "|" +
        df_key["target_community"].astype("Int64").astype(str)
    )
    key_stats = df_key.groupby("trial_key").agg(
        n=("correct", "count"),
        acc=("correct", "mean"),
        h_alias=("h_alias", "mean"),
        rt=("duration_ms", "mean"),
    ).reset_index()

    key_stats = key_stats.sort_values(["acc", "h_alias"], ascending=[True, False]).head(12)
    save_table(key_stats, "top12_hard_trials_by_key")

    # For each selected key, aggregate heatmap across all subjects for that trial_key
    # We need to pull matching raw trial dicts; build an index for speed
    raw_by_key = {}
    for t in raw_trials:
        k = f"{t.get('W')}|{t.get('curr_snapshot_id')}|{t.get('ground_truth')}|{t.get('target_community')}"
        raw_by_key.setdefault(k, []).append(t)

    for _, r in key_stats.iterrows():
        k = r["trial_key"]
        trials_k = raw_by_key.get(k, [])
        if not trials_k:
            continue
        H = heatmap_from_trials(trials_k)
        outp = os.path.join(TRIAL_HEAT_DIR, f"heat_trialkey_{k.replace('|','_')}.png")
        plot_heatmap(H, f"Aggregated heatmap | key={k} | n={len(trials_k)} | acc={r['acc']:.2f}", outp)

    # ----------------------------
    # Quick "paper-ready" tables
    # ----------------------------
    # 1) Accuracy by event x W
    pivot = df.pivot_table(index="ground_truth", columns="W", values="correct", aggfunc="mean")
    pivot = pivot.reindex(EVENTS)
    pivot_path = os.path.join(OUT_DIR, "acc_event_by_W.csv")
    pivot.to_csv(pivot_path)
    print(f"[saved] {pivot_path}")

    # 2) Mean RT by event x W
    pivot_rt = df.pivot_table(index="ground_truth", columns="W", values="duration_ms", aggfunc="mean")
    pivot_rt = pivot_rt.reindex(EVENTS)
    pivot_rt_path = os.path.join(OUT_DIR, "rt_event_by_W.csv")
    pivot_rt.to_csv(pivot_rt_path)
    print(f"[saved] {pivot_rt_path}")

    print("\n✅ Analysis complete.")
    print(f"Outputs in: {OUT_DIR}")


if __name__ == "__main__":
    main()